# Tugas Praktikum JS03 - Dataset PBL
## Evaluasi Splitting Data dan Cross Validation pada Berita Hoaks


### 1. Load dataset PBL


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/pbl_hoaks_valid_clean.csv')
print('Dimensi data:', df.shape)
df.head()


Dimensi data: (1498, 2)


,teks_bersih,label
0,presiden prabowo minta maaf ikut bop akan kemb...,1
1,video presiden prabowo akan membagikan bantuan...,1
2,pemerintah kota tangerang selatan akan legalka...,1
3,tautan rekrutmen cpns kementerian imigrasi,1
4,ramuan rebusan jahe kunyit kencur dan serai be...,1


### 2. Cek distribusi kelas target


In [2]:
print('Distribusi label target (1: Hoaks, 0: Valid):')
print(df['label'].value_counts())


Distribusi label target (1: Hoaks, 0: Valid):
label
1    749
0    749
Name: count, dtype: int64


### 3. Ekstraksi fitur TF-IDF


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(df['teks_bersih']).toarray()
y = df['label'].values

print('Dimensi matriks fitur X:', X.shape)
print('Dimensi target y:', y.shape)


Dimensi matriks fitur X: (1498, 1000)
Dimensi target y: (1498,)


### 4. Perbandingan split data 80:20 vs 70:30


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Split 80:20
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model_80 = MultinomialNB()
model_80.fit(X_train_80, y_train_80)
acc_80 = accuracy_score(y_test_80, model_80.predict(X_test_80))

# Split 70:30
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
model_70 = MultinomialNB()
model_70.fit(X_train_70, y_train_70)
acc_70 = accuracy_score(y_test_70, model_70.predict(X_test_70))

print(f'Akurasi Split 80:20 : {acc_80 * 100:.2f}%')
print(f'Akurasi Split 70:30 : {acc_70 * 100:.2f}%')


Akurasi Split 80:20 : 90.00%
Akurasi Split 70:30 : 88.67%


### 5. Evaluasi K-Fold Cross Validation (5-Fold)


In [5]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = cross_val_score(MultinomialNB(), X, y, cv=kf, scoring='accuracy')

for i, score in enumerate(scores_kf, 1):
    print(f'Fold {i}: {score * 100:.2f}%')
print(f'Rata-rata Akurasi K-Fold: {scores_kf.mean() * 100:.2f}%')


Fold 1: 87.00%
Fold 2: 88.00%
Fold 3: 87.33%
Fold 4: 86.96%
Fold 5: 91.97%
Rata-rata Akurasi K-Fold: 88.25%


### 6. Evaluasi Stratified K-Fold Cross Validation (5-Fold)


In [6]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = cross_val_score(MultinomialNB(), X, y, cv=skf, scoring='accuracy')

for i, score in enumerate(scores_skf, 1):
    print(f'Fold {i}: {score * 100:.2f}%')
print(f'Rata-rata Akurasi Stratified K-Fold: {scores_skf.mean() * 100:.2f}%')


Fold 1: 86.00%
Fold 2: 85.33%
Fold 3: 90.67%
Fold 4: 85.28%
Fold 5: 90.97%
Rata-rata Akurasi Stratified K-Fold: 87.65%
